# <font color="#418FDE" size="6.5" uppercase>**Text Features**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Convert dictionaries and text documents into scikit-learn feature matrices. 
- Configure tokenization, n-grams, vocabulary, stopwords, and frequency controls. 
- Build sparse text pipelines for classification tasks. 


## **1. Dictionary Feature Extraction**

### **1.1. Dictionary Vectorization**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_01_01.jpg?v=1783987129" width="250">



>* Turns dictionary records into feature matrices
>* Creates consistent numeric columns for estimators

>* Handles numeric, categorical, and boolean features
>* Supports irregular records and summarized text attributes

>* Sparse matrices handle many mostly empty features
>* Consistent columns keep models interpretable



In [ ]:
#@title Python Code - Dictionary Vectorization

# This example vectorizes small dictionary records.
# DictVectorizer creates numeric columns from keys.
# The output shows sparse features clearly.

from sklearn.feature_extraction import DictVectorizer
import sklearn

# Each dictionary is one observation with named attributes.
records = [
    {"city": "Paris", "plan": "basic", "purchases": 2, "discount": True},
    {"city": "Tokyo", "plan": "premium", "purchases": 5},
    {"city": "Paris", "plan": "premium", "purchases": 1, "discount": False},
]

# The vectorizer learns feature columns from dictionary keys and values.
vectorizer = DictVectorizer(sparse=True)
feature_matrix = vectorizer.fit_transform(records)

# Validate that every record became one matrix row.
if feature_matrix.shape[0] != len(records):
    raise ValueError("Each input record should become one feature row.")

# Feature names explain what each generated column means.
feature_names = vectorizer.get_feature_names_out()
print("scikit-learn version:", sklearn.__version__)
print("Matrix shape:", feature_matrix.shape)
print("Feature names:", list(feature_names))

# Dense display is acceptable here because the example is tiny.
dense_matrix = feature_matrix.toarray().astype(int)
print("First row values:", dense_matrix[0].tolist())

# New records use the same learned columns during transform.
new_record = [{"city": "Tokyo", "plan": "basic", "purchases": 3}]
new_matrix = vectorizer.transform(new_record)
print("New row values:", new_matrix.toarray().astype(int)[0].tolist())



### **1.2. Feature Hashing**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_01_02.jpg?v=1783987131" width="250">



>* Hash feature names into fixed matrix columns
>* Handle new dictionary keys without rebuilding vocabularies

>* Fast extraction for large streaming datasets
>* Sparse numeric rows without stored vocabularies

>* Hash features into fixed numeric bins
>* Consistent columns support dynamic production data



In [ ]:
#@title Python Code - Feature Hashing

# Demonstrate feature hashing for dictionary records.
# Hashing maps feature names into fixed columns.
# The output is a sparse numeric matrix.

import sklearn
from sklearn.feature_extraction import FeatureHasher

# Each dictionary represents one small user event.
events = [
    {"city=Paris": 1, "browser=Chrome": 1, "clicked": 1},
    {"city=Tokyo": 1, "browser=Firefox": 1, "clicked": 0},
    {"city=Paris": 1, "browser=Safari": 1, "clicked": 1},
]

# FeatureHasher needs no fitting or stored vocabulary.
hasher = FeatureHasher(n_features=8, input_type="dict", alternate_sign=False)
hashed_matrix = hasher.transform(events)

# Validate the fixed matrix shape before displaying results.
expected_shape = (len(events), 8)
if hashed_matrix.shape != expected_shape:
    raise ValueError("The hashed matrix shape is not as expected.")

# Convert only this tiny example to dense form for printing.
dense_matrix = hashed_matrix.toarray().astype(int)

print("scikit-learn version:", sklearn.__version__)
print("Input records:", len(events))
print("Fixed hashed matrix shape:", hashed_matrix.shape)
print("Nonzero values per row:", hashed_matrix.getnnz(axis=1).tolist())
print("Dense view for this tiny example:")
print(dense_matrix)



### **1.3. Hashing Tradeoffs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_01_03.jpg?v=1783987133" width="250">



>* Hash features without storing a vocabulary
>* Gain speed but lose readable columns

>* Hash collisions can mix unrelated feature signals
>* Choose bucket count to balance efficiency

>* Great for scalable streaming production systems
>* Harder to inspect and explain features



## **2. Text Vectorizers**

### **2.1. Count Vectorizer Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_02_01.jpg?v=1783987126" width="250">



>* Turns text into word count features
>* Bag-of-words ignores order and grammar

>* Word counts create interpretable document features
>* Usage patterns support common text classification tasks

>* Sparse matrices store mostly nonzero word counts
>* Training vocabulary limits new text representation



In [ ]:
#@title Python Code - Count Vectorizer Basics

# This example turns short texts into counts.
# CountVectorizer learns vocabulary columns from training documents.
# The output shows sparse counts and feature names.

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# These tiny documents keep the vocabulary easy to inspect.
documents = [
    "Free refund offer for loyal customers",
    "Urgent refund request after slow service",
    "Friendly service and delicious food",
]

# The vectorizer controls tokenization, stopwords, and n-grams.
vectorizer = CountVectorizer(stop_words="english", ngram_range=(1, 2))
count_matrix = vectorizer.fit_transform(documents)

# This validation confirms one row per document.
if count_matrix.shape[0] != len(documents):
    raise ValueError("The matrix should have one row per document.")

# Feature names are the learned vocabulary columns.
feature_names = vectorizer.get_feature_names_out()
selected_terms = ["refund", "service", "free refund", "slow service"]

# Convert only a few columns to a small readable table.
selected_indexes = [list(feature_names).index(term) for term in selected_terms]
small_counts = count_matrix[:, selected_indexes].toarray()
count_table = pd.DataFrame(small_counts, columns=selected_terms)

print("CountVectorizer learned", len(feature_names), "features.")
print("Matrix shape:", count_matrix.shape)
print("Sparse nonzero counts:", count_matrix.nnz)
print(count_table.to_string(index=False))



### **2.2. Token Controls**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_02_02.jpg?v=1783987124" width="250">



>* Tokenization defines model-readable text units
>* Customize tokens for domain-specific meaning

>* Choose token patterns that reduce noise
>* Preserve domain-specific meaning during preprocessing

>* Balance token rules to control feature noise
>* Inspect real tokens for domain meaning



In [ ]:
#@title Python Code - Token Controls

# This example compares token controls in vectorizers.
# Token patterns decide which text pieces count.
# The output shows changed vocabularies and matrices.

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# These short documents contain tokens defaults may drop.
documents = [
    "AI team fixed error C-3 in app_v2.",
    "Customer says app v2 is OK, but C code failed.",
    "AI support tagged #urgent for model X.",
]

# The default pattern keeps words with at least two characters.
default_vectorizer = CountVectorizer()
default_matrix = default_vectorizer.fit_transform(documents)

# This pattern keeps one-letter words and hashtag words.
custom_vectorizer = CountVectorizer(token_pattern=r"(?u)#?\b\w+\b")
custom_matrix = custom_vectorizer.fit_transform(documents)

# Compare the learned vocabularies from both token rules.
default_tokens = default_vectorizer.get_feature_names_out()
custom_tokens = custom_vectorizer.get_feature_names_out()

# Show only selected columns to keep the matrix readable.
selected_tokens = ["ai", "c", "3", "app_v2", "urgent"]
visible_tokens = []
for token in selected_tokens:
    if token in custom_tokens:
        visible_tokens.append(token)

# Convert the selected sparse columns into a small table.
selected_indexes = []
for token in visible_tokens:
    selected_indexes.append(custom_vectorizer.vocabulary_[token])

small_matrix = custom_matrix[:, selected_indexes].toarray()
small_table = pd.DataFrame(small_matrix, columns=visible_tokens)

print("scikit-learn CountVectorizer token control demo")
print("Default tokens include: " + ", ".join(default_tokens[:8]))
print("Custom tokens include: " + ", ".join(custom_tokens[:10]))
print("Default vocabulary size: " + str(len(default_tokens)))
print("Custom vocabulary size: " + str(len(custom_tokens)))
print(small_table.to_string(index=False))



### **2.3. N Gram Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_02_03.jpg?v=1783987128" width="250">



>* N grams capture short token sequences
>* Phrases preserve local meaning and word order

>* N gram range balances detail and size
>* Limit vocabulary to reduce rare noisy phrases

>* Stopword choices can preserve or break phrases
>* N grams capture useful local word patterns



In [ ]:
#@title Python Code - N Gram Features

# This example compares unigram and bigram text features.
# N grams preserve short word order patterns.
# The output shows phrase features and matrix sizes.

from sklearn.feature_extraction.text import CountVectorizer
import sklearn

# These tiny reviews make negation easy to see.
documents = [
    "not good service",
    "good service and fast shipping",
    "not fast shipping",
    "good product",
]

# Unigrams count single words only.
unigram_vectorizer = CountVectorizer(ngram_range=(1, 1))
unigram_matrix = unigram_vectorizer.fit_transform(documents)

# Unigrams plus bigrams also count adjacent word pairs.
bigram_vectorizer = CountVectorizer(ngram_range=(1, 2))
bigram_matrix = bigram_vectorizer.fit_transform(documents)

# Validate that each row still represents one document.
if unigram_matrix.shape[0] != len(documents):
    raise ValueError("The unigram matrix row count is incorrect.")

if bigram_matrix.shape[0] != len(documents):
    raise ValueError("The bigram matrix row count is incorrect.")

# Show how adding bigrams expands the feature space.
print("scikit-learn version:", sklearn.__version__)
print("Unigram matrix shape:", unigram_matrix.shape)
print("Unigram plus bigram matrix shape:", bigram_matrix.shape)

# Inspect selected phrase features that unigrams cannot represent.
feature_names = bigram_vectorizer.get_feature_names_out()
selected_features = ["not good", "good service", "fast shipping"]

# Print counts for meaningful bigram features in each document.
for feature in selected_features:
    feature_index = list(feature_names).index(feature)
    counts = bigram_matrix[:, feature_index].toarray().ravel().tolist()
    print(feature, "counts:", counts)

# Confirm that the phrase feature captures negation directly.
print("Feature 'not good' exists:", "not good" in feature_names)



## **3. TFIDF Text Pipelines**

### **3.1. TFIDF Transformation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_03_01.jpg?v=1783987136" width="250">



>* TFIDF converts text into classification features
>* Highlights distinctive words, downweights common ones

>* Sparse TFIDF matrices keep text features practical
>* Weighted terms improve classification beyond raw counts

>* Highlights distinctive terms for better generalization
>* Efficient representation, not true language understanding



In [ ]:
#@title Python Code - TFIDF Transformation

# This script demonstrates TFIDF for text classification.
# TFIDF weights distinctive words more than common words.
# A sparse pipeline predicts message categories from text.

import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# Small labeled messages keep the example easy to inspect.
texts = [
    "refund request for a damaged delivery package",
    "delivery arrived late and the package was broken",
    "please refund my order because delivery failed",
    "password reset link does not work for my account",
    "account login failed after password change",
    "please help reset my forgotten password",
    "subscription renewal charged my card twice",
    "cancel my subscription and stop future billing",
    "billing question about monthly subscription charge",
    "refund needed for missing delivery item",
    "password problem blocks account access",
    "subscription invoice has an incorrect billing amount",
]

labels = [
    "delivery_refund",
    "delivery_refund",
    "delivery_refund",
    "password_account",
    "password_account",
    "password_account",
    "subscription_billing",
    "subscription_billing",
    "subscription_billing",
    "delivery_refund",
    "password_account",
    "subscription_billing",
]

# Stratification keeps each category represented in both splits.
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels, test_size=0.25, random_state=42, stratify=labels
)

# The vectorizer learns TFIDF features only from training text.
pipeline = Pipeline(
    [
        ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2))),
        ("model", LogisticRegression(max_iter=200, random_state=42)),
    ]
)

# Fitting the pipeline transforms text and trains the classifier.
pipeline.fit(train_texts, train_labels)
predicted_labels = pipeline.predict(test_texts)
accuracy = accuracy_score(test_labels, predicted_labels)

# Inspect the sparse matrix shape and a few learned terms.
tfidf_step = pipeline.named_steps["tfidf"]
train_matrix = tfidf_step.transform(train_texts)
feature_names = tfidf_step.get_feature_names_out()

# Show the strongest TFIDF terms in one test message.
example_matrix = tfidf_step.transform([test_texts[0]])
nonzero_columns = example_matrix.nonzero()[1]
term_scores = []

for column in nonzero_columns:
    term_scores.append((feature_names[column], example_matrix[0, column]))

term_scores = sorted(term_scores, key=lambda item: item[1], reverse=True)
top_terms = [term for term, score in term_scores[:5]]

print("scikit-learn version:", sklearn.__version__)
print("Training TFIDF matrix shape:", train_matrix.shape)
print("Sparse nonzero values:", train_matrix.nnz)
print("Test accuracy:", round(accuracy, 2))
print("Example text:", test_texts[0])
print("Top TFIDF terms:", top_terms)
print("Predicted category:", predicted_labels[0])



### **3.2. Vocabulary Limits**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_03_02.jpg?v=1783987138" width="250">



>* Limit vocabulary to useful text features
>* Remove noisy terms that rarely generalize

>* Limit vocabulary by size and term frequency
>* Keep dependable terms that distinguish classes

>* Tune limits to data size and domain
>* Balance compact features with useful distinctions



In [ ]:
#@title Python Code - Vocabulary Limits

# This script demonstrates vocabulary limits in TFIDF pipelines.
# Frequency thresholds control which text features survive.
# A small classifier compares limited and unlimited vocabularies.

import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# These short messages mimic two support ticket categories.
documents = [
    "refund refund invoice payment order",
    "refund invoice charge payment cancellation",
    "delivery package tracking shipment order",
    "delivery shipment package courier tracking",
    "password login account reset access",
    "password account login reset security",
    "refund payment invoice charge order",
    "delivery tracking package shipment delay",
    "password reset login account locked",
    "refund cancellation invoice payment",
    "delivery courier package tracking",
    "password security account access",
]

# Labels mark the category each message belongs to.
labels = [
    "billing", "billing", "shipping", "shipping",
    "account", "account", "billing", "shipping",
    "account", "billing", "shipping", "account",
]

# Split first so vocabulary is learned only from training text.
train_text, test_text, train_labels, test_labels = train_test_split(
    documents, labels, test_size=0.33, random_state=42, stratify=labels
)

# This pipeline keeps every token seen in training.
wide_pipeline = Pipeline(
    [("tfidf", TfidfVectorizer()), ("model", LogisticRegression(max_iter=200))]
)

# This pipeline keeps only dependable, moderately frequent tokens.
limited_pipeline = Pipeline(
    [
        ("tfidf", TfidfVectorizer(min_df=2, max_df=0.85, max_features=8)),
        ("model", LogisticRegression(max_iter=200)),
    ]
)

# Fit both pipelines on the same training examples.
wide_pipeline.fit(train_text, train_labels)
limited_pipeline.fit(train_text, train_labels)

# Compare vocabulary size and test accuracy.
wide_vocab = wide_pipeline.named_steps["tfidf"].get_feature_names_out()
limited_vocab = limited_pipeline.named_steps["tfidf"].get_feature_names_out()
wide_accuracy = accuracy_score(test_labels, wide_pipeline.predict(test_text))
limited_accuracy = accuracy_score(test_labels, limited_pipeline.predict(test_text))

# Print concise results that show the vocabulary tradeoff.
print("scikit-learn version:", sklearn.__version__)
print("Unlimited vocabulary size:", len(wide_vocab))
print("Limited vocabulary size:", len(limited_vocab))
print("Limited vocabulary terms:", ", ".join(limited_vocab))
print("Unlimited test accuracy:", round(wide_accuracy, 2))
print("Limited test accuracy:", round(limited_accuracy, 2))



### **3.3. Sparse Text Pipeline**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_03_03.jpg?v=1783987139" width="250">



>* Convert text into compact TFIDF features
>* Keep sparse matrices efficient for classification

>* Keep training and prediction steps consistent
>* Reuse learned vocabulary for new documents

>* Evaluate transformations and classifiers together
>* Prevent leakage during fair model comparison



In [ ]:
#@title Python Code - Sparse Text Pipeline

# Build a sparse text classification pipeline.
# TFIDF keeps text features memory efficient.
# Predictions reuse the same learned vocabulary.

import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# Small labeled messages keep the example self contained.
texts = [
    "invoice payment refund billing account",
    "charged twice invoice billing refund",
    "payment failed card billing issue",
    "server error login password technical",
    "app crashes after update technical bug",
    "cannot connect network technical support",
    "cancel my subscription close account",
    "please stop service cancel plan",
    "end membership cancellation request",
    "what are your opening hours question",
    "where can I find product information",
    "general question about support options",
]

# Labels name the support category for each message.
labels = [
    "billing",
    "billing",
    "billing",
    "technical",
    "technical",
    "technical",
    "cancellation",
    "cancellation",
    "cancellation",
    "general",
    "general",
    "general",
]

# Stratification keeps every category represented in both splits.
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels, test_size=0.33, random_state=42, stratify=labels
)

# The pipeline fits TFIDF and the classifier together.
pipeline = Pipeline(
    [
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), stop_words="english")),
        ("model", LogisticRegression(max_iter=200, random_state=42)),
    ]
)

# Fitting learns the vocabulary only from training text.
pipeline.fit(train_texts, train_labels)

# The transformed matrix is sparse, not a dense array.
train_matrix = pipeline.named_steps["tfidf"].transform(train_texts)
if not hasattr(train_matrix, "nnz"):
    raise ValueError("Expected a sparse TFIDF matrix.")

# Predictions automatically use the same TFIDF settings.
predicted_labels = pipeline.predict(test_texts)
accuracy = accuracy_score(test_labels, predicted_labels)

# New messages can be classified with the same pipeline.
new_messages = [
    "refund my invoice please",
    "the app will not let me login",
]
new_predictions = pipeline.predict(new_messages)

print("scikit-learn version:", sklearn.__version__)
print("Sparse matrix shape:", train_matrix.shape)
print("Stored nonzero TFIDF values:", train_matrix.nnz)
print("Test accuracy:", round(accuracy, 2))
print("New message predictions:", list(new_predictions))



# <font color="#418FDE" size="6.5" uppercase>**Text Features**</font>


In this lecture, you learned to:
- Convert dictionaries and text documents into scikit-learn feature matrices. 
- Configure tokenization, n-grams, vocabulary, stopwords, and frequency controls. 
- Build sparse text pipelines for classification tasks. 

In the next Lecture (Lecture B), we will go over 'Images Custom'